In [ ]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# pip install --force-reinstall "transformers>=4.48.0,<5.0.0"

In [ ]:
pip install --upgrade unsloth unsloth-zoo transformers huggingface_hub

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 681.9 kB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.2
    Uninstalling transformers-4.56.2:
      Successfully uninstalled transformers-4.56.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.2

In [ ]:
!pip install -U huggingface_hub transformers datasets

  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 109.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 34.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.4 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which is incompatible.
unsloth-zoo 2026.5.4 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.

In [ ]:
import huggingface_hub

print(huggingface_hub.__version__)

1.17.0


In [ ]:
import unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
%%capture
!pip install evaluate json_repair

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from unsloth import FastLanguageModel
from datasets import Dataset
import evaluate
import numpy as np
import json

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_TOKEN = user_secrets.get_secret("WAB_DUSH")

In [ ]:
from huggingface_hub import login
import wandb

login(token=HF_TOKEN)
wandb.login(key=WANDB_TOKEN)

In [ ]:
repo_id = "dinushiTJ/nz_research_commons_inference_results"
BATCH_SIZE = 4
RETRY_ALL = False   # set True if you want to recompute everything

In [ ]:
# from huggingface_hub import list_repo_commits

# commits = list_repo_commits(
#     "dinushiTJ/nz_research_commons_inference_results",
#     repo_type="dataset",
# )

# for c in commits[:20]:
#     print(c.commit_id, c.created_at, c.title)

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    repo_id,
    split="train")

print(ds)

README.md:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3318 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse'],
    num_rows: 3318
})


In [ ]:
# from datasets import load_dataset

# ds = load_dataset(repo_id)

In [ ]:
ds

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse'],
    num_rows: 3318
})

In [ ]:
ds[0]['prompt']

'You are a specialist in classifying academic research papers by their relevance to Māori topics. Classify the academic research paper below into EXACTLY ONE category:\n- maori_origin: Substantively connected to Māori people, communities, land, resources, or knowledge\n- non_maori_origin: No meaningful Māori connection\n\nQUALIFIES AS maori_origin if it substantively involves ANY of:\n\n1. PEOPLE & COMMUNITY\n   • Māori people, whānau, hapū, or iwi as primary subject or participant group\n   • Māori health, education, economic, justice, or social outcomes\n   • Māori demographic studies where Māori outcomes are the primary focus\n\n2. CULTURE, IDENTITY & LANGUAGE\n   • Te reo Māori — acquisition, revitalisation, or documentation\n   • Māori cultural practices, tikanga, identity, whakapapa, or arts\n   • Māori oral traditions, media, or digital representation\n\n3. HISTORY, POLITICS & LAW\n   • Treaty of Waitangi, Māori land rights, Waitangi Tribunal, Treaty settlements\n   • Māori sove

In [ ]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

base_model, base_tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it-bnb-4bit",
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = HF_TOKEN, # HF Token for gated models
)

==((====))==  Unsloth 2026.5.8: Fast Gemma3 patching. Transformers: 5.9.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [ ]:
FastLanguageModel.for_inference(base_model)

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (embeddings): SiglipVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4096, 1152)
      )
      (encoder): SiglipEncoder(
        (layers): ModuleList(
          (0-26): 27 x SiglipEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (self_attn): SiglipAttention(
              (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (mlp): SiglipMLP(
              (

In [ ]:
from unsloth.chat_templates import get_chat_template

base_tokenizer = get_chat_template(base_tokenizer, chat_template="gemma-3")

In [ ]:
from time import perf_counter
import json
from json_repair import repair_json


def generate_base_response(user_message: str):
    start_time = perf_counter()

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": user_message,
                }
            ],
        }
    ]

    text = base_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = base_tokenizer(
        text=[text],
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

    end_time = perf_counter()

    raw_output = base_tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )

    # Safety: normalize output
    if isinstance(raw_output, list):
        raw_output = "".join(raw_output)

    parsed_output = None
    repaired = False

    # ---- JSON repair + parsing ----
    try:
        repaired_json = repair_json(raw_output)

        if isinstance(repaired_json, str):
            parsed_output = json.loads(repaired_json)
        else:
            parsed_output = repaired_json

        repaired = True

    except Exception as e:
        print("[WARNING]: JSON repair failed, returning raw output")
        print("[ERROR]:", repr(e))
        print("Raw output preview:", raw_output[:500])

        parsed_output = raw_output  # fallback

    return {
        "output": parsed_output,
        "raw_output": raw_output,
        "time": end_time - start_time,
        "json_ok": repaired,
    }


In [ ]:
def generate_base_response_batch(user_messages):

    messages_batch = []
    for msg in user_messages:
        messages_batch.append([
            {
                "role": "user",
                "content": [{"type": "text", "text": msg}],
            }
        ])

    texts = [
        base_tokenizer.apply_chat_template(
            m,
            add_generation_prompt=True,
            tokenize=False,
        )
        for m in messages_batch
    ]

    inputs = base_tokenizer(
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        add_special_tokens=False,
    ).to("cuda")

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

    decoded = base_tokenizer.batch_decode(
        outputs[:, inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )

    return decoded

In [ ]:
generate_base_response(ds[0]['prompt'])

{'output': {'label': 'maori_origin',
  'confidence': 0.9,
  'reason': "The research focuses on female students' educational transition experiences within Papua New Guinea (PNG). It examines factors influencing their success, including socio-cultural context, parental support, and the impact of a patriarchal society. While not directly about Māori people, the study's focus on female students transitioning within the PNG context and addressing challenges related to gender and societal norms aligns with the broader concept of supporting educational outcomes for marginalized groups, which is relevant to Māori experiences and priorities in similar contexts. The research also highlights the importance of support systems – a key factor impacting educational outcomes for Māori communities."},
 'raw_output': '```json\n{\n  "label": "maori_origin",\n  "confidence": 0.9,\n  "reason": "The research focuses on female students\' educational transition experiences within Papua New Guinea (PNG). It ex

In [ ]:
def run_base_response(example):
    try:
        prompt = example['prompt']
        response = generate_base_response(prompt)

        if isinstance(response, dict):
            response = json.dumps(response, ensure_ascii=False)

        return {"Base4BResponse": response}

    except Exception as e:
        print("ERROR:", e)
        return {"Base4BResponse": "__ERR__"}

In [ ]:
if "Base4BResponse" not in ds.column_names:
    ds = ds.add_column(
        "Base4BResponse",
        ["__ERR__"] * len(ds)
    )
    print("Added column")

In [ ]:
def should_run(example):
    if RETRY_ALL:
        return True

    v = example.get("Base4BResponse")

    return v in ["__ERR__"]

In [ ]:
remaining_indices = [
    i for i, ex in enumerate(ds)
    if should_run(ex)
]

print(f"Remaining: {len(remaining_indices)}")

Remaining: 0


Single Res

In [ ]:
# pbar = tqdm(total=len(remaining_indices), desc="Inference")

# col = list(ds["Base4BResponse"])  # ✅ FAST LOCAL COPY

# for batch_start in range(0, len(remaining_indices), BATCH_SIZE):

#     batch = remaining_indices[batch_start: batch_start + BATCH_SIZE]

#     print(f"\nBatch {batch_start // BATCH_SIZE + 1}")

#     for idx in batch:

#         try:
#             result = run_base_response(ds[idx])
#             col[idx] = result["Base4BResponse"]

#         except Exception as e:
#             print(f"Row {idx} failed: {e}")
#             col[idx] = "__ERR__"

#         pbar.update(1)

#     # -------------------------
#     # ONLY NOW update dataset
#     # -------------------------
#     ds = ds.remove_columns(["Base4BResponse"])
#     ds = ds.add_column("Base4BResponse", col)

#     print("Pushing checkpoint...")
#     ds.push_to_hub(repo_id)

# print("Done")

Batch Res

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 4  # important: tune this (4–16 usually ideal)

pbar = tqdm(total=len(remaining_indices), desc="Inference")

col = list(ds["Base4BResponse"])

for batch_start in range(0, len(remaining_indices), BATCH_SIZE):

    batch = remaining_indices[batch_start: batch_start + BATCH_SIZE]

    prompts = [ds[i]["prompt"] for i in batch]

    outputs = generate_base_response_batch(prompts)

    for idx, out in zip(batch, outputs):

        try:
            parsed = json.dumps(out, ensure_ascii=False)
            col[idx] = parsed

        except Exception as e:
            print(f"Row {idx} failed: {e}")
            col[idx] = "__ERR__"

        pbar.update(1)

    ds = ds.remove_columns(["Base4BResponse"])
    ds = ds.add_column("Base4BResponse", col)

    ds.push_to_hub(repo_id)

print("Done")

Inference:   0%|          | 1/1170 [02:13<43:19:08, 133.40s/it]Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
[datasets.arrow_dataset|WARNING]Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done


In [ ]:
err_count = sum(1 for x in ds["Base4BResponse"] if x == "__ERR__")
print("ERR count:", err_count)

ERR count: 0


In [ ]:
import pandas as pd

col = ds["Base4BResponse"]

# convert to pandas for easy analysis
s = pd.Series(col)

# normalize null-like values
null_count = s.isna().sum() + (s == "").sum()

unique_labels = s.dropna()
unique_labels = unique_labels[unique_labels != ""].unique()

print("Null / empty count:", null_count)
print("Num unique labels:", len(unique_labels))

Null / empty count: 0
Num unique labels: 3318


In [ ]:
from datasets import DatasetDict

fixed_ds = DatasetDict({
    "train": ds
})

fixed_ds.push_to_hub(repo_id)